In [166]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

# from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [167]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Q1. Embedding a query

In [6]:
query = "How does approximate nearest neighbor search work?"
v_query = model.encode(query)
v_query[0]

np.float32(-0.020581992)

### Q2. Cosine similarity


In [168]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
documents[22]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [9]:
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'
for i, doc in enumerate(documents):
    if doc['filename'] == target_filename:
        target_content = doc['content']
        target_index = i
target_index, target_content

(22,
 '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far

In [10]:
embedded_target_content = model.encode(target_content)
embedded_target_content.dot(v_query)

np.float32(0.45681334)

### Q3. Chunking and search by hand

In [169]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks), len(documents[71]['content'])

(295, 2348)

In [35]:
chunks[22:24]

[{'start': 1000,
  'content': 'to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n"""\n```\n\nThis is what grounds the answer in our data and reduces hallucinations.\n\n## The user prompt template\n\nThe user prompt template has placeholders for the question and the\ncontext:\n\n```python\nUSER_PROMPT_TEMPLATE = """\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\n## Building the context\n\nThe `context` is a formatted string with all the search results:\n\n```python\ndef build_context(search_results):\n    lines = []\n\n    for doc in search_results:\n        lines.append(doc["section"])\n        lines.append("Q: " + doc["question"])\n        lines.append("A: " + doc["answer"])\n        lines.append("")\n\n    return "\\n".join(lines).strip()\n```\n\nEach document becomes a block with the section, question, and answer.\nThis format makes it easy for the LLM to read. We turned a list of\ndicti

In [170]:
embedded_chunks = []
for chunk in chunks:
    e_c = model.encode(chunk['content'])
    embedded_chunks.append(e_c)
len(embedded_chunks), len(embedded_chunks[22])

(295, 384)

In [171]:
X = np.array(embedded_chunks)
X.shape

(295, 384)

In [51]:
scores = X.dot(v_query)

In [54]:
idx = np.argmax(scores)
scores[idx], chunks[idx]

(np.float32(0.5624412),
 {'start': 1000,
  'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because th

### Q4. Vector search with a library

In [172]:
es = Elasticsearch("http://localhost:9200")

In [ ]:
# es.indices.delete(index="homework_2_lessons", ignore_unavailable=True)

ObjectApiResponse({'acknowledged': True})

In [181]:
# Delete first in case the index existed:
es.indices.delete(index="chunk_homework_2_lessons", ignore_unavailable=True)

# Create index:
index_settings = {
    "mappings": {
        "properties": {
            "filename": {"type": "keyword"}
            ,"content": {"type": "text"}
            ,"vector": {
                "type": "dense_vector"
                ,"dims": len(X[0])
                ,"index": True
                ,"similarity": "cosine"
            }
        }
    }
}

es.indices.create(index="chunk_homework_2_lessons", body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'chunk_homework_2_lessons'})

In [180]:
len(embedded_chunks)

295

In [182]:
for i, chunk in enumerate(embedded_chunks):
    es.index(
        index="chunk_homework_2_lessons",
        id=i,
        body={
            "filename": chunks[i]["filename"]
            ,"content": chunks[i]["content"]
            ,"vector": X[i].tolist()
        }
    )

es.count(index="chunk_homework_2_lessons")['count']


295

In [185]:
query_2 = 'What metric do we use to evaluate a search engine?'
v_query_2 = model.encode(query_2)
# v_query_2

In [ ]:
def vector_search(query):
    response = es.search(
        index="chunk_homework_2_lessons"
        ,body={
            "knn": {
                "field": "vector"
                ,"query_vector": query.tolist()
                ,"k": 10 # avg number of chunks per filename is 2 so k = 2*5 (desired result)
                ,"num_candidates": 100
            }
        }
    )

    filenames  = [hit["_source"]["filename"] for hit in response["hits"]["hits"]]
    return list(dict.fromkeys(filenames)) 
        # print(hit["_score"])
        # print(hit["_source"]["filename"])
        # print("---")


In [187]:
vector_search(v_query_2)

['04-evaluation/lessons/01-intro.md',
 '04-evaluation/lessons/05-search-metrics.md',
 '01-agentic-rag/lessons/05-search.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '04-evaluation/lessons/06-search-tuning.md',
 '02-vector-search/lessons/10-next-steps.md']

### Q5. Text search vs Vector search

In [188]:
def text_search(query):
    fields =  ["content"]
    results = es.search(
        index="chunk_homework_1_lessons",
        body={
            "size": 5
            ,"query": {
                "bool": {
                    "must":  {"multi_match": {"query": query, "fields": fields}}
                }
            }
        }
    )
    return [hit["_source"]['filename'] for hit in results["hits"]["hits"]]
    # (hit["_score"], 

In [189]:
query_3 = 'How do I store vectors in PostgreSQL?'
v_query_3 = model.encode(query_3)

In [190]:
vector_search_result = vector_search(v_query_3)

In [191]:
text_search_result = text_search(query_3)


In [192]:
[i for i in vector_search_result  if i not in text_search_result]

['02-vector-search/lessons/02-embeddings.md',
 '05-monitoring/lessons/05-database.md',
 '02-vector-search/lessons/04-vector-search.md']